# Notebook 4bis — Sweep over M with multiple seeds

**Goal:** Get reliable NN distance estimates for each value of M in EFM by averaging over multiple seeds.

We test M ∈ {1, 16, 32, 64, 128, 256, 512} (M=1 corresponds to vanilla CFM) on Fashion-MNIST,
training each configuration with `N_SEEDS` different random seeds and reporting **mean ± std** NN distance.

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/YentlCollin/ClosedFlowMatching.git 2>/dev/null || true
    os.chdir('ClosedFlowMatching')
    !pip install -q -r requirements.txt
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, '..')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.data.images import get_image_dataloader, extract_all_images
from src.models.unet import SmallUNet
from src.flow_matching.sampler import ode_sample
from src.flow_matching.efm import compute_efm_target
from src.metrics.evaluation import nearest_neighbor_distance

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Config

In [ ]:
DATASET    = 'fashion_mnist'
IMAGE_SIZE = 28
BATCH_SIZE = 128
N_TRAIN    = 10000
N_EPOCHS   = 25
N_GEN      = 500   # samples generated for NN distance
N_SEEDS    = 3     # seeds per M value  (increase for more reliability, costs 3x time)
M_VALUES   = [1, 16, 32, 64, 128, 256, 512]  # M=1 ≡ vanilla CFM

## Load data

In [ ]:
train_loader = get_image_dataloader(DATASET, BATCH_SIZE, IMAGE_SIZE, train=True, n_samples=N_TRAIN)
all_train_images = extract_all_images(train_loader).to(device)
train_flat = all_train_images.view(len(all_train_images), -1)
print(f'Training images: {all_train_images.shape}')

## Training function (CFM when M=1, EFM otherwise)

In [ ]:
def train_model(train_data, M, n_epochs, batch_size, lr, seed, device):
    """Train a SmallUNet with EFM (M>=2) or CFM (M=1) for n_epochs."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SmallUNet(in_channels=1, base_channels=32, time_emb_dim=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    n = len(train_data)
    train_flat_local = train_data.view(n, -1)
    steps_per_epoch = n // batch_size

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        perm = torch.randperm(n)
        for i in range(steps_per_epoch):
            idx = perm[i * batch_size:(i + 1) * batch_size]
            x1 = train_data[idx]
            x0 = torch.randn_like(x1)

            if M == 1:
                # Vanilla CFM: t shape (B,)
                t = torch.rand(batch_size, device=device)
                x_t = (1.0 - t[:, None, None, None]) * x0 + t[:, None, None, None] * x1
                target = (x1 - x0)
                pred = model(x_t, t)
            else:
                # EFM: t shape (B, 1)
                t = torch.rand(batch_size, 1, device=device)
                x_t = (1.0 - t[:, :, None, None]) * x0 + t[:, :, None, None] * x1
                x_t_flat = x_t.view(batch_size, -1)
                x1_flat  = x1.view(batch_size, -1)
                target = compute_efm_target(x_t_flat, x1_flat, train_flat_local, t, M)
                target = target.view_as(x_t)
                pred = model(x_t, t.squeeze(-1))

            loss = ((pred - target) ** 2).mean()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg = epoch_loss / steps_per_epoch
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'    Epoch {epoch+1:2d}/{n_epochs} — loss: {avg:.4f}')

    return model


def compute_nn_dist(model, train_flat, n_gen, image_size, device):
    model.eval()
    with torch.no_grad():
        gen = ode_sample(model, n_gen, (1, image_size, image_size), n_steps=100, device=device)
    gen_flat = gen.view(n_gen, -1).cpu()
    return nearest_neighbor_distance(gen_flat, train_flat.cpu()).mean().item()

## Sweep M × seeds

For each M and each seed: train → compute NN distance. This is the long cell.

In [ ]:
# results[M] = list of NN distances, one per seed
results = {M: [] for M in M_VALUES}

total_runs = len(M_VALUES) * N_SEEDS
run = 0

for M in M_VALUES:
    label = 'CFM (M=1)' if M == 1 else f'EFM M={M}'
    for seed in range(N_SEEDS):
        run += 1
        print(f'\n[{run}/{total_runs}] {label}  seed={seed}')
        model = train_model(
            all_train_images, M=M,
            n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
            lr=2e-4, seed=seed, device=device
        )
        nn_d = compute_nn_dist(model, train_flat, N_GEN, IMAGE_SIZE, device)
        results[M].append(nn_d)
        print(f'    → NN dist: {nn_d:.4f}')

print('\nDone!')

## 4.7 Nearest-neighbor distance (memorization metric) — mean ± std over seeds

In [ ]:
means = np.array([np.mean(results[M]) for M in M_VALUES])
stds  = np.array([np.std(results[M])  for M in M_VALUES])
labels = ['CFM\n(M=1)'] + [f'EFM\nM={M}' for M in M_VALUES[1:]]

print('Results summary:')
for M, mu, sigma in zip(M_VALUES, means, stds):
    tag = 'CFM' if M == 1 else f'EFM M={M}'
    print(f'  {tag:12s}  NN dist = {mu:.4f} ± {sigma:.4f}')

# Color: grey for CFM, gradient of blue for EFM
colors = ['#888888'] + [plt.cm.Blues(0.4 + 0.6 * i / (len(M_VALUES) - 2))
                        for i in range(len(M_VALUES) - 1)]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(M_VALUES))
bars = ax.bar(x, means, yerr=stds, color=colors, edgecolor='white',
              capsize=5, error_kw=dict(elinewidth=1.5, ecolor='black'))

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean NN distance (↑ = more generalization)')
ax.set_title(f'NN distance vs M — {DATASET}  ({N_SEEDS} seeds, mean ± std)')

# Annotate each bar with its mean value
for xi, mu in zip(x, means):
    ax.text(xi, mu + stds[xi] + 0.05, f'{mu:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig4bis_nn_distance_sweep.pdf', bbox_inches='tight')
plt.show()

## Raw results table

In [ ]:
print(f'{"M":>8}  {"seeds":>40}  {"mean":>8}  {"std":>8}')
print('-' * 72)
for M, mu, sigma in zip(M_VALUES, means, stds):
    seeds_str = '  '.join(f'{v:.3f}' for v in results[M])
    print(f'{M:>8}  {seeds_str:>40}  {mu:>8.4f}  {sigma:>8.4f}')